In [ ]:
import random
import math
from scipy.stats import chi2
import numpy as np
from tabulate import tabulate
# importamos t-student(t) y normal(norm)
from scipy.stats import t,norm
from typing import Any, Callable, List, Optional, Tuple
from math import factorial

def generar_binomial(n, p):
    return np.random.binomial(n, p)

def P_binomial(x, n, p):
    return (math.factorial(n) / (math.factorial(x)* math.factorial(n-x))) * pow(p, x) * pow(1-p, n-x)

def generar_exponencial(media):
    return np.random.exponential(media)

def P_exponencial(x, media):
    return 1 - math.exp(-(x*(1/media)))

def generar_poisson(media):
    return np.random.poisson(media)

def P_poisson(x, media):
    return math.exp(-media) * pow(media, x) / factorial(x)

def generar_normal(mu, sigma):
    return np.random.normal(mu, sigma)

##def P_normal(x, mu, sigma):
 ##   return (1 / (sigma * math.sqrt(2 * math.pi))) * math.exp(-pow((x - mu), 2) / (2 * pow(sigma, 2)))

def estimador_T_Pearson(N_list: list[int], N: int, p_list: list[float]):

    """
    Estima T con el test de Pearson con aproximación chi cuadrada.

    Parametros

    N_list: Lista de las frecuencias observadas.
    N: Tamaño de la muestra.
    p_list: Lista de las frecuencias teoricas.
    """

    assert len(N_list) == len(p_list)
    assert sum(N_list) == N

    T = 0
    for ni, pi in zip(N_list, p_list):
        T += pow((ni - (N*pi)), 2)/(N*pi)
    return T


def simulacion_p_valor_binomiales(N, probs, T, n_sim):

    """
    Simulación general del p-valor all in one.

    Parámetros

    N: Tamaño de la muestra.
    probs: Lista de las frecuencias teoricas.
    T: Estadístico original.
    n_sim: Número de simulaciones.
    """

    n = 0
    cont = 0
    while n <= n_sim:
        # calcular N_i
        N_list = []
        sum_probs = 0
        for prob in probs:
            N_i = np.random.binomial(n = N - sum(N_list), p = prob / (1-sum_probs))
            N_list.append(N_i)
            sum_probs += prob

        # Calcular T
        t = 0
        for i in range(len(probs)):
            value = pow(N_list[i]-(N*probs[i]), 2) / (N*probs[i])
            t += value

        if t > T:
            cont += 1
        n += 1

    return cont/n_sim

#Kolmogorov smirnoff discreto con parámetros especificados.
def estadistico_d(F, m):

    """
    Calcula el estadístico de Kolmogorov-Smirnoff usando la función de masa en vez de la empírica.

    Parámetros:
    F: Función de masa.
    m: Lista de valores aleatorios.
    """

    muestra = m
    muestra.sort()

    d = 0
    n = len(muestra)
    for i in range(1, len(muestra)+1):
        x = muestra[i-1]
        d = max(d, ((i)/n)-F(x), F(x)-((i-1)/n)) #Versión final del estadístico que no usa la empírica.

    return d


def estadistico_smirnov(sample, F):
    sample = sorted(sample)

    n = len(sample)

    d = max(max((i+1)/n - F(x), F(x) - i/n) for i, x in enumerate(sample))

    return d


Ejercicio 1)

De acuerdo con la teoría genética de Mendel, cierta planta de guisantes debe producir flores
blancas, rosas o rojas con probabilidad 1/4, 1/2 y 1/4, respectivamente. Para verificar experimentalmente
la teoría, se estudió una muestra de 564 guisantes, donde se encontró que 141 produjeron flores blancas,
291 flores rosas y 132 flores rojas. Aproximar el p-valor de esta muestra:


In [ ]:
def simulacion_p_valor(N, n_sim):

    """

    Simula el p-valor usando uniformes para generar las frecuencias observadas.
    Para eso genera muestras de tamaño N de la distribución F y calcula el estadístico T.
    El p-valor va a ser la proporción de valores de T obtenidos que superen al t original de la muestra original.

    Parámetros
    N: Tamaño de la muestra.
    n_sim: Número de simulaciones.

    """

    n = 0
    cont = 0

    while n <= n_sim:

        #VALORES DE PROBABILIDADES TEORICAS DEL EJERCICIO 1
        #P1 = 1/4, P2 = 1/2, P3 = 1/4

        N1 = np.random.binomial(n=N, p = (1 / 4))
        N2 = np.random.binomial(n=N - N1, p = (1 / 2) / (1 - (1 / 4)))
        N3 = np.random.binomial(n=N - N1 - N2, p = (1 / 4) / (1 - (1 / 4) - (1 / 2)))

        T = (N1 - N * (1 / 4)) ** 2 / (N * (1 / 4))
        T += (N2 - N * (1 / 2)) ** 2 / (N * (1 / 2))
        T += (N3 - N * (1 / 4)) ** 2 / (N * (1 / 4))

        if T >= 0.8617021: #Si supera el valor del estadístico original que es 0.8617021 según la fórmula de a)
            cont += 1
        n += 1

    return cont / n_sim #Proporción de valores que superaron el estadístico original de a)


print(f"El valor del estimador T original es: {estimador_T_Pearson([141, 291, 132], 564, [0.25, 0.5, 0.25])}")
print(f"El valor del p-valor estimado mediante Pearson es: {chi2.sf(x=0.8617021, df=2)}") #df = grados de libertad = cantidad posible de valores - 1
print(f"El valor del p-valor simulado es: {simulacion_p_valor(564, 100000)}")

p_valor = simulacion_p_valor_binomiales(N=564, probs=[0.25, 0.5, 0.25], T = 0.8616, n_sim=100_000,)

print(f"El valor del p-valor estimado con la fórmula cheta es: {p_valor}")


El valor del estimador T original es: 0.8617021276595745
El valor del p-valor estimado mediante Pearson es: 0.6499557144687855
El valor del p-valor simulado es: 0.65387
El valor del p-valor estimado con la fórmula cheta es: 0.65402


Ejercicio 2: Para verificar que cierto dado no estaba trucado, se registraron 1000 lanzamientos, resultando
que el número de veces que el dado arrojó el valor i (i = 1,2,3,4,5,6) fue, respectivamente, 158, 172, 164,
181, 160, 165. Aproximar el p−valor de la prueba: “el dado es honesto”

Hipótesis nula: el dado es honesto.

Hipótesis alternativa: el dado es un mentiroso cara de oso.

In [ ]:
#a) Con el test de pearson.

lista_observaciones = [158, 172, 164, 181, 160, 165]

lista_probabilidades = [1/6, 1/6, 1/6, 1/6, 1/6, 1/6] #Asumiendo que es honesto y todos los resultados son equiprobables.

N = 1000

T = estimador_T_Pearson(lista_observaciones, N, lista_probabilidades)

print(f"El valor del estimador T original es: {T}")

print(f"El valor del p-valor estimado mediante Pearson es: {chi2.sf(x=T, df=5)}") #df = grados de libertad = cantidad posible de valores - 1

print(f"El valor del p-valor estimado mediante simulación (100.000 simulaciones) es: {simulacion_p_valor_binomiales(N, lista_probabilidades, T, 100000)}")

El valor del estimador T original es: 2.18
El valor del p-valor estimado mediante Pearson es: 0.8237195392577814
El valor del p-valor estimado mediante simulación (100.000 simulaciones) es: 0.82339


Ejercicio 3: Calcular una aproximación del p−valor de la hipótesis: “Los siguientes 10 números son aleatorios”:

In [ ]:
def F_x(x):
    assert 0 <= x <= 1
    return x


def estimacion_p_valor_ej3(n, d, n_sim):
    p_valor = 0
    for _ in range(n_sim):
        uniformes = np.random.uniform(0, 1, n) #Para que la hipótesis sea cierta, los números tienen que venir de una uniforme.
        uniformes.sort()
        d_j = estadistico_d(F_x, uniformes)
        if d_j > d:
            p_valor += 1

    return p_valor/n_sim

muestras = [0.12, 0.18, 0.06, 0.33, 0.72, 0.83, 0.36, 0.27, 0.77, 0.74]
print(f"muestras            {muestras}")
d_calc = estadistico_d(F=F_x, m=muestras)
print(f"estadistico d       {d_calc}")
p_valor = estimacion_p_valor_ej3(n=len(muestras), d=d_calc, n_sim=10_000)
print(f"estimacion p_valor  {p_valor}")

muestras            [0.12, 0.18, 0.06, 0.33, 0.72, 0.83, 0.36, 0.27, 0.77, 0.74]
estadistico d       0.24
estimacion p_valor  0.5305


Ejercicio 4: Calcular una aproximación del p−valor de la hipótesis: “Los siguientes 13 valores provienen
de una distribución exponencial con media 50,0”:

Valores: 86.0, 133.0, 75.0, 22.0, 11.0, 144.0, 78.0, 122.0, 8.0, 146.0, 33.0, 41.0, 99.0

In [ ]:
#La F que va a usar Kolmogorov para comparar.
def exponencial_ej4(x, media):
  return 1 - math.exp(-(x*media))

def estimacion_p_valor_ej4(n, d, n_sim):
    p_valor = 0
    for _ in range(n_sim):
        exponenciales = np.random.exponential(50,n) #Para que la hipótesis sea cierta, los números tienen que venir de una exponencial de media 1/50.
        exponenciales.sort() #Lista de muestras ordenadas para pasarle a la F.
        d_j = estadistico_d(lambda x: 1 - math.exp(-x * (1/50)), exponenciales)
        if d_j > d:
            p_valor += 1

    return p_valor/n_sim

muestra = [86.0, 133.0, 75.0, 22.0, 11.0, 144.0, 78.0, 122.0, 8.0, 146.0, 33.0, 41.0, 99.0]
muestra.sort()

def f(x):
    return exponencial_ej4(x=x, media=1/50)

d = estadistico_d(f, muestra)
print(f"estadistico d       {d}")

p_valor_estimado = estimacion_p_valor_ej4(n=len(muestra), d=d, n_sim=100000)
print(f"estimacion p_valor  {p_valor_estimado}")

estadistico d       0.3922544552361856
estimacion p_valor  0.02557


Ejercicio 5: Calcular una aproximación del p−valor de la prueba de que los siguientes datos corresponden a una distribución binomial con parámetros (n = 8, p), donde p no se conoce:

Muestra: 6, 7, 3, 4, 7, 3, 7, 2, 6, 3, 7, 8, 2, 1, 3, 5, 8, 7.


In [ ]:
#Primero estimamos los parámetros desconocidos y despues calculamos el estadístico de Kolmogorov.
def media_muestral(m):
    return sum(m) / len(m)

def p_binomial(x, n, p):
    return math.comb(n, x) * pow(p, x) * pow(1-p, n-x)

def sim_p_valor(t_sample, n_bin, p_bin, N_sample, n_sim):

    p_valor = 0
    n = 0
    while n <= n_sim:

        values = np.random.binomial(n=n_bin, p=p_bin, size=N_sample)
        ocurrencias = [0 for _ in range(n_bin+1)]
        for i in values:
            ocurrencias[i] += 1

        pi = sum(values)/len(values) / 8
        probs = [p_binomial(i, n_bin, pi) for i in range(n_bin+1)]
        Ti = estimador_T_Pearson(ocurrencias, N_sample, probs)
        if Ti >= t_sample:
            p_valor += 1

        n += 1

    return p_valor / n_sim


# Calcular una aproximación del p−valor de la prueba de que los siguientes datos corresponden
# a una distribución binomial con parámetros (n = 8, p), donde p no se conoce

muestra = [6, 7, 3, 4, 7, 3, 7, 2, 6, 3, 7, 8, 2, 1, 3, 5, 8, 7]

n = 8

# aproximamos el parametro p, como E(X)=np, tenemos que p=E(X)/n
p_aprox = media_muestral(m=muestra)/n
print(p_aprox)

# luego, utilizamos este parametro para el test de Pearson

def p_x(x):
    return p_binomial(x, 8, p_aprox)

ocurrencias = [0 for _ in range(n+1)]

for elem in muestra:
    ocurrencias[elem] += 1

print(ocurrencias)

t = estimador_T_Pearson(ocurrencias, sum(ocurrencias), [p_x(i) for i in range(n+1)])

grados_libertad = len(ocurrencias) - 1 - 1 #ocurrencias menos la cantidad de parametros desconocidos menos 1.

p_valor = chi2.sf(x=t, df=grados_libertad)

p_valor_sim = sim_p_valor(t_sample=t, n_bin=8, p_bin=p_aprox, N_sample=len(muestra), n_sim=10_000)

print(f"aproximacion p                      {p_aprox}")
print(f"estimador T                         {t}")
print(f"aproximacion p-valor con chi2       {p_valor:.5f}")
print(f"simulacion p-valor                  {p_valor_sim}")

0.6180555555555556
[0, 1, 2, 4, 1, 1, 2, 5, 2]
aproximacion p                      0.6180555555555556
estimador T                         31.499330934155324
aproximacion p-valor con chi2       0.00005
simulacion p-valor                  0.0091


Ejercicio 6: Un escribano debe validar un juego en cierto programa de televisión. El mismo consiste en hacer girar una rueda y obtener un premio según el sector de la rueda que coincida con una aguja.

Hay 10 premios posibles, y las áreas de la rueda para los distintos premios, numerados del 1 al 10, son respectivamente:

    31%, 22%, 12%, 10%, 8%, 6%, 4%, 4%, 2%, 1%

Los premios con número alto  son mejores que los premios con número bajo.

El escribano hace girar la rueda hasta que se cansa, y anota cuántas veces sale
cada sector.

Los resultados, para los premios del 1 al 10, respectivamente, son:

    188, 138, 87, 65, 48, 32, 30, 34, 13, 2


a) Construya una tabla con los datos disponibles.

b) Diseñe una prueba de hipótesis para determinar si la rueda es justa.

c) Defina el p-valor a partir de la hipótesis nula.

d) Calcule el p-valor bajo la hipótesis de que la rueda es justa, usando la aproximación chi cuadrado.

e) Calcule el p-valor bajo la hipótesis de que la rueda es justa, usando una simulación.


Hipótesis nula: La rueda es justa.

Hipótesis alternativa: La rueda no es justa.

La idea del test va a ser ver si los datos obtenidos al simular el p-valor y el estadístico T difieren de los esperados por más de un determinado umbral de tolerancia.

El p-valor es una medida estadística que nos ayuda a determinar si los resultados observados son consistentes con la hipótesis nula.
En este caso, la hipótesis nula podría establecer que la rueda funciona correctamente y que los premios aparecen con la frecuencia esperada según sus áreas asignadas.
Para calcular el p-valor, se puede realizar una prueba de bondad de ajuste, como la prueba chi-cuadrado, comparando las frecuencias observadas con las esperadas.
Las frecuencias esperadas se obtienen multiplicando el número total de giros por la probabilidad teórica de cada premio:

- Frecuencia esperada = Probabilidad * Total de giros
- Total de giros = 188 + 138 + 87 + 65 + 48 + 32 + 30 + 34 + 13 + 2 = 637


In [ ]:
#a) Genero la tabla.
probabilidades = [x/100 for x in [31, 22, 12, 10, 8, 6, 4, 4, 2, 1]]
ocurrencias = [188, 138, 87, 65, 48, 32, 30, 34, 13, 2]

table = {

    "Premio": list(range(1, 11)),
    "Areas (%)": probabilidades,
    "Ocurrencias": ocurrencias

        }

print("Tabla\n", tabulate(table, headers="keys"))



Tabla
   Premio    Areas (%)    Ocurrencias
--------  -----------  -------------
       1         0.31            188
       2         0.22            138
       3         0.12             87
       4         0.1              65
       5         0.08             48
       6         0.06             32
       7         0.04             30
       8         0.04             34
       9         0.02             13
      10         0.01              2


In [ ]:
#Probabilidad de masa de X, donde X es el sector de la rueda que es seleccionado.
def p_x():
    F = [0.0, 0.31, 0.53, 0.65, 0.75, 0.83, 0.89, 0.93, 0.97, 0.99, 1]
    U = random.random()
    i = 0
    while U > F[i]:
        i += 1
    assert i != 0
    return i


def simulacion_p_valor(t, gen, probs, n_sim):

    p_valor = 0
    for _ in range(n_sim):

        # calcular N_i
        muestra = []

        ocurrencias = [0 for _ in range(10)]

        #Calculo las ocurrencias teóricas
        for _ in range(637):
            elem = gen() - 1
            muestra.append(elem)
            ocurrencias[elem] += 1

        #Calculo el estimador T
        T = 0
        N = sum(ocurrencias)

        for i in range(len(ocurrencias)):
            T += pow(ocurrencias[i] - (N*probs[i]), 2) / (N*probs[i])

        if T >= t:
            p_valor += 1

    return p_valor / n_sim

N = sum(ocurrencias)

t = estimador_T_Pearson(ocurrencias, N, probabilidades)
df = 10-1

print(f"Estimador t         {t}")
print(f"p-valor Pearson     {chi2.sf(x=t, df=df)}")
print(f"p-valor simulado    {simulacion_p_valor(t, p_x, probabilidades, 10000)}")

Estimador t         9.810370888711903
p-valor Pearson     0.3660538998868262
p-valor simulado    0.3709


Ejercicio 7: Generar los valores correspondientes a 30 variables aleatorias exponenciales independientes, cada una con media 1.

Luego, en base al estadístico de prueba de Kolmogorov-Smirnov, aproxime el
p−valor de la prueba de que los datos realmente provienen de una distribución exponencial con media 1.


In [ ]:
def exponencial(x):
    return 1 - math.exp(-x)

#Smirnov con exponencial
def p_valor_smirnoff_exp(d, N,  n_sim):

    def F_x(x):
        assert 0 <= x <= 1
        return x

    p_valor = 0
    for _ in range(n_sim):

        exponenciales = np.random.exponential(1, N)
        exponenciales.sort()
        d_j = estadistico_d(lambda x: 1-math.exp(-x), exponenciales)

        if d_j >= d:
            p_valor += 1

    return p_valor/n_sim

#Genero las 30 exponenciales reales.
muestra = np.random.exponential(1, size=30)
N=30
d_stat = estadistico_d(exponencial, muestra)
print(f"estadistico d           {d_stat}")

# estimamos el p-valor
p_valor = p_valor_smirnoff_exp(d=d_stat, N=N, n_sim=10_000)

print(f"estiamcion p-valor      {p_valor}")
p_valor = p_valor_smirnoff_exp(d=d_stat, N=N, n_sim=10_000)
print(f"estimacion p-valor exp  {p_valor}")

estadistico d           0.18576237901293285
estiamcion p-valor      0.2173
estiamcion p-valor exp  0.2179


Ejercicio estdístico d parcial 2024:

In [ ]:
def d_statistic_unif(n):
    """
    Estadistico KS para una muestra uniforme(0,1) de tamaño n.

    Parameters
    ----------
    n : int
        Number of random samples to generate from the uniform distribution.
        Tamanio de la muestra original.
    """
    n = int(n)
    u = np.random.uniform(0,1,n)
    u = np.sort(u)
    d = 0
    for i in range(1,n+1):
      d = max(d, i / n - u[i-1], u[i-1] - (i-1)/n)
    return d

def d_statistic_norm(muestra,loc=0,scale=1):
    """
    Estadistico KS para una muestra dada suponiendo una distribuacion
    norm(loc,scale).

    Parameters
    ----------
    muestra : array_like
          Muestra dada.
    loc : float
        Mean of the normal distribution.
    scale : float
        Standard deviation the normal distribution.
    """
    n = len(muestra)
    d = 0
    muestra_ordenada = np.sort(muestra.copy())
    norm_cdf = norm(loc,scale).cdf(muestra_ordenada)
    for i in range(1,n+1):
      d = max(d, i / n - norm_cdf[i-1], norm_cdf[i-1] - (i-1)/n)
    return d

def estimacion_p_valor(n, d, n_sim):
    p_valor = 0
    for _ in range(n_sim):
        uniformes = np.random.uniform(0, 1, n) #Para que la hipótesis sea cierta, los números tienen que venir de una uniforme.
        uniformes.sort()
        d_j = d_statistic_unif(n)
        if d_j > d:
            p_valor += 1
    return p_valor/n_sim

muestra = [1.628, 1.352, 1.420, 1.594, 1.614, 1.692, 1.8, 1.924, 2.132]
estadistico_d_teorico = estadistico_d(muestra, 1.691, 0.058)
print(f"estadistico d teorico       {estadistico_d_teorico}")

pvalor = estimacion_p_valor(n=len(muestra), d=estadistico_d_teorico, n_sim=10000)
print(f"estimacion p-valor          {pvalor}")

estadistico d teorico       0.41686182216424533
estimacion p-valor          0.059


In [ ]:
"""
El test de hipótesis es:
  Hipótesis nula: La muestra proviene de una distribución binomial de parámetros n=3 y p donde p se estima como
  la media muestral de la muestra obtenida (porque la media es el estimador insesgado de p y bla bla)
  La media es: (0*490 + 1*384 + 2*111 + 3*15)/1000*3 = 0.217
  Hipótesis alternativa: La muestra no proviene de una distribución binomial de parámetros 3 y 0.217.

  El p-valor a partir de la hipótesis nula es una medida de semejanza de la distribución que se desea verificar con la distribución con la cual
  se la compara.
  Si este pvalor es menor a un cierto umbral de tolerancia, se rechaza la hipótesis nula y se acepta la alternativa.
  De lo contrario decimos que no tenemos evidencia suficiente para rechazar la hipótesis nula con ese umbral de tolerancia.

"""

def estimador_T_Pearson(N_list: list[int], N: int, p_list: list[float]):

    """
    Estima T con el test de Pearson con aproximación chi cuadrada.

    Parametros
                N_list: Lista de las frecuencias observadas.
                N: Tamaño de la muestra.
                p_list: Lista de las frecuencias teoricas.
    """

    assert len(N_list) == len(p_list)
    assert sum(N_list) == N
    T = 0
    for ni, pi in zip(N_list, p_list):
        T += pow((ni - (N*pi)), 2)/(N*pi)
    return T

def prob_binomial(x, p): #con n = 3 hardcodeado
    return (math.factorial(3) / (math.factorial(x)* math.factorial(3-x))) * pow(p, x) * pow(1-p, 3-x)

def generar_binomial(n, p):
    return np.random.binomial(n, p)

def p_value_sim(nIter, n, muestra, t):

    p_0 = 0.217
    t_0 = t
    succ = 0

    for _ in range(nIter):
        #Genero una muestra de binomiales en cada simulación.
        sample = []
        for i in range(n):
            sample.append(generar_binomial(3, p_0))

        #Nueva probabilidad basada en la muestra (estimada con la media)
        p = sum(sample) / (n * 3)

        frecuencias = [0] * 4

        #Frecuencias de la muestra
        for m in sample:
            frecuencias[m] += 1

        #Con las frecuencia y probabilidad de esta muestra, calculo las nuevas teóricas.

        probs_teos = [prob_binomial(i, p) for i in range(4)]
        t = estimador_T_Pearson(frecuencias, sum(frecuencias), probs_teos)

        #La proporción de valores de t que se pasen del original es la estimacion del p-valor.
        if t >= t_0:
            succ += 1

    return succ / nIter


probabilidades_teoricas = [prob_binomial(i,0.217) for i in range(4)]
ocurrencias = [490, 384, 111, 15]
estimador_t = estimador_T_Pearson(ocurrencias, sum(ocurrencias), probabilidades_teoricas)

print(f"Estimador T: {estimador_t}")
pvalor = chi2.sf(x=estimador_t, df=len(ocurrencias)-1-1)
print(f"p-valor estimado con pearson: {pvalor}")

pvalor_sim = p_value_sim(nIter=10000, n=1000, muestra=ocurrencias, t=estimador_t)
print(f"p-valor estimado con simulacion: {pvalor_sim}")

#No se rechaza

Estimador T: 3.0181185228436638
p-valor estimado con pearson: 0.22111789428006515
p-valor estimado con simulacion: 0.2162


Ej2 parcial 2024

In [ ]:
def montecarlo_0_inf(func: Callable, threshold):

    def h(u):
        return (1 / (u**2)) * func((1 / u) - 1)

    media = func(random.random())
    varianza = 0
    n = 1
    intervalo = math.sqrt(varianza / n)

    while n <= 100 or intervalo > threshold:
        sim = h(random.random())
        media_ant = media
        media = media_ant + (sim - media_ant) / (n + 1)
        varianza_ant = varianza
        varianza = (1 - (1 / n)) * varianza + (n + 1) * (media - media_ant) ** 2
        n += 1
        intervalo = math.sqrt(varianza / n)
    return media, math.sqrt(varianza), intervalo, n

def funcion(x):
  return x/(1+x**4)

zalpha_2 = 1.95 # alpha = 0.05
L = 2 * 0.001
d = L / (2 * zalpha_2)

media, desvio, intervalo, n = montecarlo_0_inf(funcion,d)
print(f"media: {media}")
print(f"desvio: {desvio}")
print(f"intervalo: {intervalo}")


media: 0.7854800745711469
desvio: 0.662296206885451
intervalo: 0.0005128204206053567
